# 01 · Profile every file before you trust any of it

**Produces:** `data/clean/profile_files.csv` (one row per file) and `data/clean/profile_columns.csv` (one row per column per table family), plus the first entries of the decisions log.

**Needs:** the data pack in `data/raw/`.

The rule this notebook enforces: a count is only as true as the step that recorded it. Before any join, you know every file's shape, encoding, date format, id format, null rate and date range, and you have written down what surprised you.

## CONFIG

In [2]:
import sys, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
from kalpa import loaders, models, checks, logs
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
CONFIG = {
    "brief": "A",
    "group": "4",
    "raw": ROOT / "data" / "raw",
    "clean": ROOT / "data" / "clean",
    "period_this": ("2026-01-01", "2026-06-30"),
    "period_last": ("2025-01-01", "2025-06-30"),
    "seed": 20261019,
    "workers": None,          # None = all cores
}
CONFIG["clean"].mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(CONFIG["seed"])
print(pd.__version__, np.__version__)

3.0.6 2.5.4


## 1 · MAP: what is in the pack

The pack is an export of six source systems. Each family has its own shape. The first move is to list every file with its size and family, so you know what you are dealing with before you read a byte of it.

In [3]:
files = loaders.list_pack(CONFIG["raw"])
print(f"{len(files)} files, {files['bytes'].sum()/1e6:.1f} MB")
files.groupby("family").agg(files=("path", "count"), mb=("bytes", lambda b: round(b.sum()/1e6, 1))).sort_values("mb", ascending=False)

381 files, 76.7 MB


,files,mb
family,,
patients,2,28.3
legacy,103,17.3
results,109,11.3
invoices,7,6.5
payments,28,6.2
collections,109,4.9
diagrams,9,1.5
nova,6,0.7
campaigns,2,0.1


### CHECK

In [4]:
import json, hashlib
manifest = json.load(open(CONFIG["raw"] / "MANIFEST.json"))
listed = {f["path"]: f for f in manifest["files"]}
missing = [p for p in listed if not (CONFIG["raw"] / p).exists()]
checks.check("every file in MANIFEST.json is present", len(missing) == 0, f"{len(missing)} missing")
# spot-check three hashes rather than all 300: enough to know the download is intact
sample = list(listed)[:: max(1, len(listed)//3)][:3]
ok = all(hashlib.sha256((CONFIG["raw"] / p).read_bytes()).hexdigest() == listed[p]["sha256"] for p in sample)
checks.check("spot-checked SHA-256 match the manifest", ok, sample)

PASS  every file in MANIFEST.json is present   0 missing
FAIL  spot-checked SHA-256 match the manifest   ['README.md', 'bookings/legacy/bengaluru/bookings_2025-03.csv.gz', 'results/chennai/results_2026-03.csv.gz']


False

## 2 · DO: read one legacy file the naive way, and watch it go wrong

A legacy monthly export is Latin-1, dates are day first, and the vendor appended files so some carry the header twice. Read one naively first. The failure is the lesson.

In [5]:
legacy_files = sorted((CONFIG["raw"] / "bookings" / "legacy").rglob("*.csv.gz"))
biggest = lambda paths: max(paths, key=lambda p: p.stat().st_size)   # the largest file of a family is the honest sample; the smallest is often an edge
print("legacy_files: ",)
one = biggest(legacy_files)
print(one.name)
try:
    naive = pd.read_csv(one)           # default encoding is UTF-8
    print("read without error:", naive.shape)
except UnicodeDecodeError as err:
    print("UnicodeDecodeError:", str(err)[:120])
print("The file we are taking is ", one)

legacy_files: 
bookings_2026-02.csv.gz
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 3541: invalid continuation byte
The file we are taking is  /home/chaitu/Kalpa_Health_Project_Group_No_4/data/raw/bookings/legacy/bengaluru/bookings_2026-02.csv.gz


In [6]:
naive = pd.read_csv(one, encoding="latin-1")
print(naive.dtypes)
print()
print("rows whose booking_id is literally the header:", (naive["booking_id"] == "booking_id").sum())
print("slot_at parsed as:", type(naive["slot_at"].iloc[0]).__name__, "->", naive["slot_at"].iloc[0])

booking_id        str
created_at        str
slot_at           str
city              str
clinic_id         str
channel           str
source            str
patient_id        str
patient_name      str
test_code         str
test_price      int64
home_fee        int64
dtype: object

rows whose booking_id is literally the header: 0
slot_at parsed as: str -> 13/02/2026 09:45


> **Interview angle.** Why did `pd.read_csv` fail on the first attempt, and why would guessing the encoding be a mistake?
>
> *What a good answer covers:* The file is Latin-1; UTF-8 is the default. Guessing by trying encodings until one works can silently mangle accented characters into different, valid characters. Find the encoding from the source system or the vendor and state it in the loader.

## 3 · DO: the proper reader

`loaders.read_legacy_file` reads with the right encoding, drops repeated header lines and counts them, parses the day-first dates with an explicit format, strips the ids, and declares dtypes so a 200,000-row file costs a tenth of the memory it would as plain objects.

In [7]:
t0 = time.time()
df = loaders.read_legacy_file(one)
print(f"{len(df):,} booking-test rows in {time.time()-t0:.2f}s; repeated header lines dropped: {df.attrs['repeated_header_lines']}")
print(df.dtypes)
print(f"memory: {df.memory_usage(deep=True).sum()/1e6:.1f} MB  (naive object frame: {naive.memory_usage(deep=True).sum()/1e6:.1f} MB)")
df.head(3)

19,602 booking-test rows in 0.12s; repeated header lines dropped: 0
booking_id              string
created_at      datetime64[us]
slot_at         datetime64[us]
city                  category
clinic_id             category
channel               category
source                category
patient_id              string
patient_name            string
test_code             category
test_price               Int32
home_fee                 Int32
source_file                str
system                     str
dtype: object
memory: 6.8 MB  (naive object frame: 11.9 MB)


,booking_id,created_at,slot_at,city,clinic_id,channel,source,patient_id,patient_name,test_code,test_price,home_fee,source_file,system
0,BK-BLR-202602-092121,2026-02-13 05:39:00,2026-02-13 09:45:00,Bengaluru,BLR-01,home,app,PBLR0024184,Sneha Fernandes,PK04,2499,0,bookings_2026-02.csv.gz,legacy
1,BK-BLR-202602-092122,2026-02-25 04:57:00,2026-02-25 13:41:00,Bengaluru,BLR-01,clinic,app,PBLR0060190,Nisha Menon,T016,160,0,bookings_2026-02.csv.gz,legacy
2,BK-BLR-202602-092123,2026-01-31 04:21:00,2026-02-01 10:16:00,Bengaluru,BLR-03,home,app,PBLR0054249,Aarav Singh,T007,550,0,bookings_2026-02.csv.gz,legacy


> **Interview angle.** What does declaring `dtype={'city': 'category'}` change, and when is `category` the wrong choice?
>
> *What a good answer covers:* A category stores each distinct value once and an integer code per row, so memory drops and groupby gets faster. It is wrong for high-cardinality columns such as ids, where the codes table is as large as the data, and it needs care on concat when the two frames have different category sets.

## 4 · DO: profile every family with the same function

One function, applied to every table family: rows, distinct keys, null rate per column, min and max date, and the share of ids matching the expected format. Run it on one file per family here; notebook 02 runs the full load.

In [8]:
def profile(df: pd.DataFrame, name: str, key: str | None = None) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({"table": name, "column": col, "dtype": str(s.dtype), "null_rate": round(float(s.isna().mean()), 4),
                     "distinct": int(s.nunique(dropna=True)),
                     "min": s.min() if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) else "",
                     "max": s.max() if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_datetime64_any_dtype(s) else ""})
    out = pd.DataFrame(rows)
    if key is not None:
        print(f"{name}: {len(df):,} rows, {df[key].nunique():,} distinct {key}  ->  {len(df)/max(1, df[key].nunique()):.2f} rows per key")
    return out

prof = [profile(df, "bookings_legacy", "booking_id")]
nova_files = sorted((CONFIG["raw"] / "bookings" / "nova").rglob("*.jsonl.gz"))
if nova_files:
    nv = loaders.read_nova_file(biggest(nova_files))
    prof.append(profile(nv, "bookings_nova", "booking_id"))
coll = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "collections").glob("*.csv.gz"))), parse_dates=["collected_at"])
prof.append(profile(coll, "collections", "booking_id"))
res = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "results").rglob("*.csv.gz"))))
prof.append(profile(res, "results", "booking_id"))
inv = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "invoices").glob("*.csv.gz"))), dtype={"amount": "string"})
prof.append(profile(inv, "invoices", "invoice_id"))
pay = loaders.read_simple_csv(biggest(list((CONFIG["raw"] / "payments").glob("gateway_*.csv.gz"))))
prof.append(profile(pay, "payments", "payment_id"))
pat = pd.read_csv(CONFIG["raw"] / "patients" / "patients.csv.gz")
prof.append(profile(pat, "patients", "patient_id"))
profile_columns = pd.concat(prof, ignore_index=True)
profile_columns

bookings_legacy: 19,602 rows, 8,344 distinct booking_id  ->  2.35 rows per key
bookings_nova: 13,966 rows, 5,903 distinct booking_id  ->  2.37 rows per key
collections: 8,161 rows, 8,120 distinct booking_id  ->  1.01 rows per key
results: 16,310 rows, 7,084 distinct booking_id  ->  2.30 rows per key
invoices: 83,630 rows, 83,630 distinct invoice_id  ->  1.00 rows per key
payments: 28,758 rows, 28,758 distinct payment_id  ->  1.00 rows per key
patients: 309,217 rows, 309,217 distinct patient_id  ->  1.00 rows per key


,table,column,dtype,null_rate,distinct,min,max
0,bookings_legacy,booking_id,string,0.0,8344,,
1,bookings_legacy,created_at,datetime64[us],0.0,7414,2026-01-26 20:51:00,2026-03-01 17:56:00
2,bookings_legacy,slot_at,datetime64[us],0.0,6751,2026-02-01 06:00:00,2026-02-28 19:56:00
3,bookings_legacy,city,category,0.0,1,,
4,bookings_legacy,clinic_id,category,0.0,6,,
...,...,...,...,...,...,...,...
60,patients,patient_name,str,0.0,1200,,
61,patients,phone,str,0.0,309200,,
62,patients,dob,str,0.0,24456,,
63,patients,city,str,0.0,6,,


### SEE: the things a profile shows that a head() does not

- `rows per key` above 1.0 on a bookings file means the grain is booking-test, not booking. Count bookings with `nunique`, never with `len`.
- An `amount` column that profiles as `string` holds thousands separators. Sum it and pandas raises, or worse, concatenates.
- A `created_at_epoch` column that profiles as integer with a min near 1.7 billion is Unix seconds.
- A null rate of 0.003 on `test_code` is small until you remember that every one of those rows is a test you cannot price.

In [8]:
id_ok = df["booking_id"].str.match(loaders.ID_PATTERNS["legacy"].pattern)
checks.check("every legacy booking id matches BK-CCC-YYYYMM-NNNNNN after strip", bool(id_ok.all()), f"{(~id_ok).sum()} do not")
checks.check("legacy slot_at parsed for every row", bool(df["slot_at"].notna().all()), f"{df['slot_at'].isna().sum()} NaT")
checks.check("invoice amount is text with separators (so we know to parse it)", pd.api.types.is_string_dtype(inv["amount"]), inv["amount"].iloc[0])
checks.check("payments amount is in paise (integer, large)", pd.api.types.is_integer_dtype(pay["amount_paise"]) and pay["amount_paise"].median() > 1000, pay["amount_paise"].median())

FAIL  every legacy booking id matches BK-CCC-YYYYMM-NNNNNN after strip   29 do not
PASS  legacy slot_at parsed for every row   0 NaT
PASS  invoice amount is text with separators (so we know to parse it)   1,499.00
PASS  payments amount is in paise (integer, large)   125000.0


True

## 5 · DO: the identifier audit

Every table that should join on an id gets its id format checked against a pattern. Mismatches are not errors yet; they are a list of things to decide about in notebook 03.

In [9]:
audit = {
    "legacy booking_id (raw, before strip)": naive["booking_id"].astype("string").str.match(r"^BK-[A-Z]{3}-\d{6}-\d{6}$").mean(),
    "collections booking_id": coll["booking_id"].astype("string").str.match(r"^(BK-[A-Z]{3}-\d{6}-\d{6}|nova_[a-z]{3}_\d{7})$").mean(),
    "results booking_id": res["booking_id"].astype("string").str.match(r"^(BK-[A-Z]{3}-\d{6}-\d{6}|nova_[a-z]{3}_\d{7})$").mean(),
    "invoice_id": inv["invoice_id"].astype("string").str.match(r"^INV-\d{4}-\d{6}$").mean(),
    "payment notes_invoice_ref, exact format": pay["notes_invoice_ref"].astype("string").str.match(r"^INV-\d{4}-\d{6}$").mean(),
    "payment notes_invoice_ref, after normalisation": loaders.normalise_invoice_ref(pay["notes_invoice_ref"]).str.match(r"^INV-\d{4}-\d{6}$").mean(),
}
pd.Series(audit, name="share_matching_pattern").round(4).to_frame()

,share_matching_pattern
"legacy booking_id (raw, before strip)",0.9877
collections booking_id,0.9978
results booking_id,0.9977
invoice_id,1.0000
"payment notes_invoice_ref, exact format",0.7051
"payment notes_invoice_ref, after normalisation",1.0000


> **Interview angle.** A payment reference is 'inv-2026-000123 ' with a trailing space. Is normalising it a cleaning decision or a parsing decision, and who should be able to audit it?
>
> *What a good answer covers:* It is a decision: you are asserting that this string identifies that invoice. It is reversible and low-risk, so it belongs in the decisions log with its rule and the count of rows it changed. A transposed digit is different: it cannot be repaired without guessing, so it stays unmatched and is listed.

## 6 · SUM: write the profile out and log the first decisions

In [11]:
files.to_csv(CONFIG["clean"] / "profile_files.csv", index=False)
profile_columns.to_csv(CONFIG["clean"] / "profile_columns.csv", index=False)
print("wrote profile_files.csv and profile_columns.csv")
# Example of recording a decision the moment it is made. Uncomment and fill in when you have decided.
# logs.decision("bookings_legacy", "header line repeated mid-file", "drop", "an append artefact, not data",
#               rows_affected=df.attrs["repeated_header_lines"], effect_on_headline="none (not rows)", owner="<name>", n=1)
checks.summary()

wrote profile_files.csv and profile_columns.csv
1 of 2 checks passed


,check,passed,detail
0,every file in MANIFEST.json is present,True,0 missing
1,spot-checked SHA-256 match the manifest,False,"['README.md', 'bookings/legacy/bengaluru/booki..."


## Before you move on

- Which table is the source of truth for a count of tests, and why that one? Write it in `docs/approach-log.md` row 1.
- Which join will be hardest, and what did the id audit tell you about it?
- What did you expect the row counts to be before you looked? How far off were you?

In [ ]:
logs.approach(part="Which table to count tests from",
              considered="Count booking rows; sum n_tests; count result rows",
              chosen="Sum n_tests from valid bookings",
              why="A booking can contain multiple tests. Results measure tests run, while the plan is for booked test volume.",
              would_change="If the plan were based on completed lab tests, use results instead.If the plan were based on completed lab tests, use results instead.",
              happened="The representative legacy file had 19,602 rows for 8,344 bookings; the Nova sample had 13,966 rows for 5,903 bookings. Counting rows would not count bookings.",
              owner="chaitanya", n=1)